# RingVoz — Preparación de datos y selección de variables

**Integrantes:** José Becerra · Marleny Guiral Marín · Geraldine Suárez  
**Responsable de este notebook:** José Becerra

Modelo descriptivo: integración, calidad, ingeniería de características y **selección de variables** sobre el ARFF crudo (24 columnas) para producir el dataset final (9 columnas, sin leakage) que consumen el modelo de clustering y el modelo predictivo.

- Entrada: `mining_data_clean.arff` (24 columnas, 89.890 filas).
- Salida: `dataset_fraude_ringvoz_class_last.arff` (9 columnas: 8 predictoras + `is_fraud` al final).
- Regla de construcción de `is_fraud`: `Result = Declined` y `respreasoncode ∈ {27, 35, 65}` y `respsubcode ∈ {1, 2}` y no es transacción de prueba.
- Códigos de fraude: 27 = AVS Fraud (dirección no coincide), 35 = tarjeta en lista negra, 65 = exceso de intentos (bot).

In [ ]:
import re
from datetime import datetime

BASE_PATH   = '/content/'
INPUT_PATH  = BASE_PATH + 'mining_data_clean.arff'
OUTPUT_PATH = BASE_PATH + 'dataset_fraude_ringvoz_class_last.arff'

FRAUD_REASON_CODES = {27, 35, 65}
FRAUD_SUB_CODES    = {1, 2}

print('Entrada :', INPUT_PATH)
print('Salida  :', OUTPUT_PATH)

In [ ]:
DAYS_EN = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

def parse_arff(filepath):
    attribute_names, data_lines, in_data = [], [], False
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            s = line.strip()
            if s.upper() == '@DATA':
                in_data = True; continue
            if not in_data:
                m = re.match(r'@[Aa][Tt][Tt][Rr][Ii][Bb][Uu][Tt][Ee]\s+["\']?(\w+)["\']?', s)
                if m: attribute_names.append(m.group(1))
            elif s and not s.startswith('%'):
                data_lines.append(s)
    return attribute_names, data_lines

def split_arff_row(row):
    out, cur, in_q, qc = [], '', False, None
    for ch in row:
        if in_q:
            if ch == qc: in_q = False
            cur += ch
        else:
            if ch in ('"', "'"):
                in_q, qc = True, ch; cur += ch
            elif ch == ',':
                out.append(cur.strip()); cur = ''
            else:
                cur += ch
    out.append(cur.strip()); return out

def parse_date_v2(s):
    s = s.strip().strip('"').strip("'").strip()
    s = re.sub(r'[+-]\d{2}:\d{2}$', '', s).strip()
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%Y-%m-%d %H:%M'):
        try:
            dt = datetime.strptime(s, fmt)
            return str(dt.hour), DAYS_EN[dt.weekday()]
        except ValueError:
            continue
    return '?', '?'

def calcular_is_fraud(result, reason_raw, sub_raw, xtest_raw):
    try:
        if result.strip('\'" ').lower() != 'declined': return '0'
        if int(float(reason_raw.strip('\'" '))) not in FRAUD_REASON_CODES: return '0'
        if int(float(sub_raw.strip('\'" ')))    not in FRAUD_SUB_CODES:    return '0'
        if xtest_raw.strip('\'" ').lower() not in ('false','0','no'):       return '0'
        return '1'
    except (ValueError, AttributeError):
        return '0'

def quote_nominal(v):
    if not v or v == '?': return '?'
    v = v.strip('\'" ').strip()
    return f"'{v}'" if v else '?'

## Exploración del ARFF crudo

In [ ]:
attr_names, data_lines = parse_arff(INPUT_PATH)
print(f'Atributos: {len(attr_names)} | Filas: {len(data_lines):,}')
for i, name in enumerate(attr_names):
    print(f'  [{i:02d}] {name}')

**Selección de variables (criterio de relevancia + ausencia de leakage).**

| Grupo | Columnas | Decisión |
|---|---|---|
| Identificadores | `Id`, `Ref`, `Account_Name`, `leadid`, `pmntGwTransId`, `userid` | Descartar (ruido, riesgo de overfitting) |
| Señales del gateway (leakage) | `Result`, `respreasoncode`, `respsubcode`, `x_test_request` | Usar solo para construir `is_fraud`, después descartar |
| Predictoras conservadas | `Account_Type`, `Amount`, `Transaction_Type`, `Merchant`, `Transaction_Use`, `pmntMethodId`, `Date_Created` | Conservar (la fecha se descompone en `hour_of_day` + `day_of_week`) |
| Excluidas por nulos/redundancia | `Zip_Code` (18 % nulos > umbral 15 %), `amExcludingTax` (r=0,999 con `Amount`), `transusedetail` (constante 0), `CRM_User`, `Year`, `postresult`, `postsent` | Descartar |

Resultado de la selección: **8 predictoras + 1 etiqueta = 9 columnas**.

In [ ]:
def idx(name):
    try: return attr_names.index(name)
    except ValueError: return None

i_result   = idx('Result')
i_acct     = idx('Account_Type')
i_amount   = idx('Amount')
i_txntype  = idx('Transaction_Type')
i_merchant = idx('Merchant')
i_txnuse   = idx('Transaction_Use')
i_pmnt     = idx('pmntMethodId')
i_date     = idx('Date_Created')
i_reason   = idx('respreasoncode')
i_subcode  = idx('respsubcode')
i_xtest    = idx('x_test_request')
print('Índices mapeados OK')

## Construcción del dataset final

En una sola pasada se filtran las transacciones de prueba, se construye `is_fraud`, se derivan `hour_of_day` y `day_of_week` desde `Date_Created`, y se escribe el ARFF con `is_fraud` como **última columna** (convención Weka: `class_last`).

In [ ]:
output_rows = []
fraud_count = legit_count = test_skipped = 0

for row in data_lines:
    vals = split_arff_row(row)
    def g(i):
        if i is None or i >= len(vals): return '?'
        v = vals[i].strip().strip('\'"')
        return v if v else '?'

    xtest = g(i_xtest)
    if xtest.lower() in ('true','1','yes'):
        test_skipped += 1; continue

    is_fraud = calcular_is_fraud(g(i_result), g(i_reason), g(i_subcode), xtest)
    if is_fraud == '1': fraud_count += 1
    else: legit_count += 1

    date_raw = vals[i_date].strip() if i_date is not None and i_date < len(vals) else '?'
    hour, dow = parse_date_v2(date_raw)

    output_rows.append(','.join([
        quote_nominal(g(i_acct)),       # [0] Account_Type
        g(i_amount),                    # [1] Amount
        quote_nominal(g(i_txntype)),    # [2] Transaction_Type
        quote_nominal(g(i_merchant)),   # [3] Merchant
        quote_nominal(g(i_txnuse)),     # [4] Transaction_Use
        g(i_pmnt),                      # [5] pmntMethodId
        hour,                           # [6] hour_of_day
        quote_nominal(dow),             # [7] day_of_week
        quote_nominal(is_fraud),        # [8] is_fraud (al final)
    ]))

DEST_HEADER = """@relation dataset_fraude_ringvoz

@attribute Account_Type     {'Retail','Corporate','Merchant','Not Defined'}
@attribute Amount           NUMERIC
@attribute Transaction_Type {'Recharge from IVR','Recharge from APM','Mobile App Recharge','Automatic Recharge','Merchant Recharge','RingVoz Website Recharge','Recharge iSMS','Activation Recharge','Monthly Recurrent Charge','Twixtext Recharge'}
@attribute Merchant         {'Authorize.Net 2 [Nr. 1469355]','Stripe','Cash','Wallet'}
@attribute Transaction_Use  {'Balance Recharge','Gift Card Recharge','Internet Nauta Recharge','Intl Mobile Recharge','Invoice Payment','Multiple Recharge','Plan Payment','Twixtext Recharge'}
@attribute pmntMethodId     NUMERIC
@attribute hour_of_day      NUMERIC
@attribute day_of_week      {'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'}
@attribute is_fraud         {'0','1'}

@data"""

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write(DEST_HEADER + '\n')
    for r in output_rows: f.write(r + '\n')

total = fraud_count + legit_count
print(f'Registros: {total:,} | Legítimas: {legit_count:,} | Fraudes: {fraud_count} '
      f'({fraud_count/total*100:.3f} %) | Pruebas excluidas: {test_skipped}')

**Resultado de la construcción.** 89.890 transacciones válidas con 17 fraudes (0,019 %). El desbalance es extremo y condiciona todo el modelado supervisado posterior: descarta accuracy como métrica, obliga a usar PR-AUC y pesos de clase, y desaconseja oversampling sintético (SMOTE con 17 positivos no es confiable).

## Descripción estadística

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

amounts, pmnt_ids, hours = [], [], []
acct_counts, txntype_counts, merchant_counts, txnuse_counts, dow_counts = {}, {}, {}, {}, {}
fraud_counts = {'0': 0, '1': 0}

for row in output_rows:
    vals = split_arff_row(row)
    def gv(i): return vals[i].strip().strip('\'"') if i < len(vals) else '?'
    acct_counts[gv(0)]     = acct_counts.get(gv(0), 0) + 1
    txntype_counts[gv(2)]  = txntype_counts.get(gv(2), 0) + 1
    merchant_counts[gv(3)] = merchant_counts.get(gv(3), 0) + 1
    txnuse_counts[gv(4)]   = txnuse_counts.get(gv(4), 0) + 1
    dow_counts[gv(7)]      = dow_counts.get(gv(7), 0) + 1
    fraud_counts[gv(8)]    = fraud_counts.get(gv(8), 0) + 1
    try: amounts.append(float(gv(1)))
    except: pass
    try: pmnt_ids.append(float(gv(5)))
    except: pass
    try: hours.append(float(gv(6)))
    except: pass

print('fraud_counts:', fraud_counts)
print('Account_Type:', acct_counts)

In [ ]:
fig = plt.figure(figsize=(20, 24))
fig.suptitle('Descripción estadística — RingVoz', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

ax = fig.add_subplot(gs[0, 0])
ax.pie([fraud_counts.get('0',0), fraud_counts.get('1',0)],
       labels=['Legítima (0)','Fraude (1)'], colors=['#2E75B6','#FF6B6B'],
       autopct='%1.2f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
ax.set_title('Distribución is_fraud', fontweight='bold')

ax = fig.add_subplot(gs[0, 1])
ax.barh(list(acct_counts.keys()), list(acct_counts.values()), color='#2E75B6', edgecolor='white')
ax.set_title('Account Type', fontweight='bold')

ax = fig.add_subplot(gs[0, 2])
ax.hist([a for a in amounts if 0 < a <= 200], bins=40, color='#2E75B6', edgecolor='white')
media = sum(amounts)/len(amounts)
ax.axvline(media, color='red', linestyle='--', label=f'Media: ${media:.2f}'); ax.legend(fontsize=8)
ax.set_title('Amount (≤ $200)', fontweight='bold')

ax = fig.add_subplot(gs[1, :])
tt = sorted(txntype_counts.items(), key=lambda x: -x[1])
ax.barh([k for k,_ in tt], [v for _,v in tt], color='#1F3864', edgecolor='white')
ax.set_title('Transaction Type', fontweight='bold')

ax = fig.add_subplot(gs[2, 0])
tu = sorted(txnuse_counts.items(), key=lambda x: -x[1])
ax.pie([v for _,v in tu], labels=[k for k,_ in tu], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':1}, textprops={'fontsize':7})
ax.set_title('Transaction Use', fontweight='bold')

ax = fig.add_subplot(gs[2, 1])
ax.bar(range(len(merchant_counts)), list(merchant_counts.values()), color='#2E75B6', edgecolor='white')
ax.set_xticks(range(len(merchant_counts)))
ax.set_xticklabels([l.replace('Authorize.Net 2 [Nr. 1469355]','Auth.Net') for l in merchant_counts], fontsize=8)
ax.set_title('Merchant', fontweight='bold')

ax = fig.add_subplot(gs[2, 2])
hc = {}
for h in hours: hc[int(h)] = hc.get(int(h), 0) + 1
ax.bar(sorted(hc), [hc[k] for k in sorted(hc)], color='#1F3864', edgecolor='white')
ax.set_title('Hora del día', fontweight='bold'); ax.set_xticks(range(0,24,2))

ax = fig.add_subplot(gs[3, 0:2])
orden = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
ax.bar(orden, [dow_counts.get(d,0) for d in orden], color='#2E75B6', edgecolor='white')
ax.set_title('Día de la semana', fontweight='bold')

ax = fig.add_subplot(gs[3, 2])
pc = {}
for p in pmnt_ids: pc[int(p)] = pc.get(int(p), 0) + 1
etq = {0:'Wallet/0', 1:'Crédito', 2:'Débito', 3:'Prepago'}
ax.bar([etq.get(k,str(k)) for k in sorted(pc)], [pc[k] for k in sorted(pc)],
       color='#1F3864', edgecolor='white')
ax.set_title('pmntMethodId', fontweight='bold')

plt.savefig(BASE_PATH + 'estadistica_descriptiva_ringvoz.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

**Hallazgos de la descripción estadística.**

- **Desbalance extremo de `is_fraud`** (0,019 %): condiciona el uso de PR-AUC y `class_weight` en el modelado.
- **`Account_Type` dominado por Retail** (≈ 97 %): Corporate y Merchant son minoría, las predicciones del modelo para esas categorías tendrán menor cobertura muestral.
- **`Amount` concentrado entre $0–$200**, media ≈ $26,67, mediana ≈ $23,99: distribución típica de recargas de móvil; valores > $500 son raros.
- **`Merchant` dominado por Authorize.Net**: una sola pasarela explica casi todo el volumen; Wallet, Stripe y Cash son colas pequeñas.
- **`Transaction_Type`** liderado por *Mobile App Recharge* y *RingVoz Website Recharge*.
- **`Transaction_Use`** liderado por *Intl Mobile Recharge*, consistente con el modelo de negocio de recargas internacionales.
- **Distribución horaria** concentrada en horario diurno; **día de la semana** con mayor actividad entre lunes y viernes.

## Análisis de nulos sobre el dataset final

In [ ]:
vars_finales = [
    (0,'Account_Type','Categórica'), (1,'Amount','Numérica'),
    (2,'Transaction_Type','Categórica'), (3,'Merchant','Categórica'),
    (4,'Transaction_Use','Categórica'), (5,'pmntMethodId','Numérica'),
    (6,'hour_of_day','Numérica'), (7,'day_of_week','Categórica'),
    (8,'is_fraud','Binaria'),
]
tot = len(output_rows)
print(f'Registros: {tot:,}  |  Umbral nulos: 15 %')
print(f"{'Variable':<20}{'Tipo':<14}{'Nulos':>8}{'%':>10}   Acción")
for i, n, t in vars_finales:
    nulos = sum(1 for row in output_rows
                if split_arff_row(row)[i].strip() in ('?','',"''",'""'))
    pct = nulos / tot * 100
    if pct > 15: a = 'ELIMINAR'
    elif pct > 0 and t == 'Numérica': a = 'Imputar (media/mediana)'
    elif pct > 0: a = 'Imputar (moda)'
    else: a = 'OK — sin nulos'
    print(f'{n:<20}{t:<14}{nulos:>8,}{pct:>9.2f}%   {a}')

**Resultado del análisis de nulos.** Las 9 columnas del dataset final tienen **0 % de nulos**. Esto es consecuencia del filtrado previo: la única variable con nulos significativos en el ARFF crudo (`Zip_Code`, 18,11 %) fue eliminada por superar el umbral del 15 %, y los demás campos fueron preservados solo cuando se obtuvo un valor válido durante la construcción. No se requiere imputación.

## Correlaciones — redundancia entre predictoras numéricas

In [ ]:
amounts_c, pmnt_c, hours_c, fraud_c = [], [], [], []
for row in output_rows:
    vals = split_arff_row(row)
    def gv(i): return vals[i].strip().strip('\'"') if i < len(vals) else '?'
    try: amounts_c.append(float(gv(1)))
    except: amounts_c.append(0.0)
    try: pmnt_c.append(float(gv(5)))
    except: pmnt_c.append(0.0)
    try: hours_c.append(float(gv(6)))
    except: hours_c.append(0.0)
    try: fraud_c.append(float(gv(8)))
    except: fraud_c.append(0.0)

M = np.corrcoef([amounts_c, pmnt_c, hours_c, fraud_c])
labels = ['Amount','pmntMethodId','hour_of_day','is_fraud']

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(M, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_title('Correlación de Pearson entre numéricas + is_fraud', fontweight='bold')
plt.colorbar(im, ax=ax)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{M[i,j]:.3f}', ha='center', va='center',
                color='white' if abs(M[i,j]) > 0.5 else 'black')
fig.tight_layout()
fig.savefig(BASE_PATH + '3_6_redundancia_ringvoz.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'r(Amount, pmntMethodId)      = {M[0,1]:+.4f}')
print(f'r(Amount, hour_of_day)       = {M[0,2]:+.4f}')
print(f'r(pmntMethodId, hour_of_day) = {M[1,2]:+.4f}')

**Resultado — redundancia.** Todos los pares de predictoras numéricas tienen `|r| < 0,10`. **Ninguna predictora numérica es redundante.** Se conservan las tres (`Amount`, `pmntMethodId`, `hour_of_day`).

## Correlaciones — irrelevancia frente a `is_fraud`

In [ ]:
vars_num = ['Amount','pmntMethodId','hour_of_day']
corr_fraud = [M[3,0], M[3,1], M[3,2]]

fig, ax = plt.subplots(figsize=(7,4))
colors = ['#2E75B6' if abs(c) > 0.1 else '#FF6B6B' for c in corr_fraud]
ax.barh(vars_num, [abs(c) for c in corr_fraud], color=colors, edgecolor='white')
ax.axvline(0.1, color='red', linestyle='--', label='Umbral 0,10'); ax.legend()
ax.set_title('|r| con is_fraud', fontweight='bold')
for i, c in enumerate(corr_fraud):
    ax.text(abs(c)+0.0005, i, f'{c:+.4f}', va='center')
fig.tight_layout()
fig.savefig(BASE_PATH + '3_7_irrelevancia_ringvoz.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

for v, c in zip(vars_num, corr_fraud):
    print(f'{v:<15} r = {c:+.4f}')

**Resultado — irrelevancia.** Las tres predictoras numéricas presentan `|r| < 0,10` con `is_fraud`. **Se conservan igualmente** porque (1) Pearson solo detecta relación lineal y los modelos basados en árboles (Random Forest, XGBoost) capturan interacciones y no-linealidades que esta métrica no ve; (2) con solo 17 positivos sobre 89.890 transacciones, el coeficiente de Pearson tiene varianza altísima y no es un criterio confiable de descarte; (3) el análisis univariado lineal no es base suficiente para excluir variables en problemas de detección de fraude con desbalance extremo.

## Conclusiones

- **Dataset final generado**: `dataset_fraude_ringvoz_class_last.arff` con 89.890 registros y 9 columnas (8 predictoras + `is_fraud` al final). 17 fraudes (0,019 %).
- **Selección de variables**: de las 24 columnas originales se conservaron 8 predictoras eliminando IDs, columnas con > 15 % nulos (`Zip_Code`), redundancias (`amExcludingTax` r=0,999 con `Amount`), constantes (`transusedetail`) y señales del gateway que producirían leakage (`Result`, `respreasoncode`, `respsubcode`).
- **Ingeniería de características**: `Date_Created` descompuesta en `hour_of_day` y `day_of_week`; `is_fraud` construida por regla de negocio combinando rechazo + códigos de fraude + exclusión de pruebas.
- **Calidad de datos**: 0 nulos en el dataset final, no se requiere imputación.
- **Correlaciones**: ninguna redundancia entre numéricas; baja correlación lineal con `is_fraud` se justifica por desbalance extremo y se compensa con modelos no lineales en la fase de modelado.
- **Salida hacia las siguientes etapas**: este ARFF es el input directo de `clustering_ringvoz.ipynb` (modelo descriptivo no supervisado) y `modelado_riesgo_ringvoz.ipynb` (modelo predictivo supervisado).
